# Data Cleaning — KKBox subsample

Raw data: [Kaggle KKBox churn data](https://www.kaggle.com/competitions/kkbox-churn-prediction-challenge/data).

**Observation period:** 2015-01-01 – 2017-02-28

**Eligible users**
1. `registration_init_time` within the observation window
2. At least one row in `transactions`; `first_transaction_date` = min `transaction_date`

**Then** random sample 1,000 users, extract rows, and merge `transactions` + `transactions_v2` and `user_logs` + `user_logs_v2`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from kkbox_subsample import KKBoxSubsampleConfig, merge_subsample_tables, run_pipeline

In [2]:
DATA_DIR = Path("kkbox-churn-prediction-challenge")

cfg = KKBoxSubsampleConfig(
    raw_dir=DATA_DIR,
    out_dir=Path("kkbox-subsample"),
    obs_start=20150101,
    obs_end=20170228,
    n_sample=1000,
    random_state=42,
)

sample_users = run_pipeline(cfg)


Scanning registrations: kkbox-churn-prediction-challenge/members_v3.csv
  Registered in observation window: 4,194,804
Scanning first transaction dates (2 file(s))...
  With at least one transaction: 900,214
  With registration + transaction: 900,214
Saved eligible cohort -> kkbox-subsample/eligible_users.csv
Sampling 1000 users...
Saved sample metadata -> kkbox-subsample/sample_users.csv
Saved sample ids -> kkbox-subsample/sample_msno.csv
Extracting subsample tables (chunked)...
  Extracting members...
    -> kkbox-subsample/subsample/members.csv (1,000 rows)
  Extracting transactions...
    -> kkbox-subsample/subsample/transactions.csv (6,270 rows)
  Extracting user_logs...
    -> kkbox-subsample/subsample/user_logs.csv (106,397 rows)
  Extracting transactions_v2...
    -> kkbox-subsample/subsample/transactions_v2.csv (573 rows)
  Extracting user_logs_v2...
    -> kkbox-subsample/subsample/user_logs_v2.csv (8,259 rows)
  members: 1,000 rows
  transactions: 6,270 rows
  user_logs: 106,

## Build panel data (user × month since joining)

**Periods:** `user_tenure` = 0, 1, 2, … calendar months since `first_transaction_date`.

**Subscription status** (each user-month):

| Status | Meaning |
|--------|---------|
| `active` | Paid membership covers at least part of the month (`txn_date`–`membership_expire_date`). |
| `paused` | Lapsed after a spell (no renewal within 30 days of expiry) but **resubscribes** later in the window. |
| `churn` | First lapse with **no** return before 2017-02-28; absorbing (panel ends that month). |

**Other fields:** `cum_hours`, `price`, `auto_renewal` (forward-filled within the panel).

In [11]:
SUBSAMPLE_DIR = Path("kkbox-subsample")
OBS_END = pd.Timestamp("2017-02-28")
RENEWAL_GRACE_DAYS = 30

users = pd.read_csv(SUBSAMPLE_DIR / "sample_users.csv", dtype={"msno": str})
transactions = pd.read_csv(SUBSAMPLE_DIR / "subsample/transactions_merged.csv", dtype={"msno": str})
logs = pd.read_csv(SUBSAMPLE_DIR / "subsample/user_logs_daily.csv", dtype={"msno": str})

users["join_date"] = pd.to_datetime(users["first_transaction_date"].astype(str), format="%Y%m%d")
transactions["txn_date"] = pd.to_datetime(transactions["transaction_date"].astype(int).astype(str), format="%Y%m%d")
transactions["expire_date"] = pd.to_datetime(
    transactions["membership_expire_date"].astype(int).astype(str), format="%Y%m%d"
)
logs["log_date"] = pd.to_datetime(logs["date"].astype(int).astype(str), format="%Y%m%d")

users.head()

,msno,registration_init_time,first_transaction_date,join_date
0,OpmUG8E57VDqDWHzpmL7es+jg9iLbPrqycTnuqA/S0k=,20161229,20161229,2016-12-29
1,2XJHzNBpZvAF/kBBsYA1S/L00KTZZEc18sfH3YxmuUo=,20160531,20161113,2016-11-13
2,kOJwBkxRTJJaddoB0uDzHzOD4ST8hu7Xt+o0x5CTVYo=,20160228,20161110,2016-11-10
3,Qxm5X3WD2Rh5Xi8iUWPLVOhtTzH0CFQk2pKBtcZmw50=,20150401,20150410,2015-04-10
4,Zk/pd0WT52ZlSRx6bR8wK6mWR7+xKW4uyPe1QaU4XWk=,20150127,20150204,2015-02-04


In [12]:
def months_since_join(join_date: pd.Series | pd.Timestamp, event_date: pd.Series | pd.Timestamp) -> pd.Series:
    """Integer months since join (calendar month difference)."""
    join_s = join_date if isinstance(join_date, pd.Series) else pd.Series([join_date])
    event_s = event_date if isinstance(event_date, pd.Series) else pd.Series([event_date])
    return (event_s.dt.year - join_s.dt.year) * 12 + (event_s.dt.month - join_s.dt.month)


def calendar_month_bounds(join_date: pd.Timestamp, tenure: int) -> tuple[pd.Timestamp, pd.Timestamp]:
    start = join_date.replace(day=1) + pd.DateOffset(months=tenure)
    end = start + pd.DateOffset(months=1) - pd.Timedelta(days=1)
    return start, end


def _month_overlaps(ms: pd.Timestamp, me: pd.Timestamp, start: pd.Timestamp, end: pd.Timestamp) -> bool:
    return ms <= end and me >= start


def _renewed_within_grace(spells: pd.DataFrame, expire: pd.Timestamp, lapse_confirm: pd.Timestamp) -> bool:
    return (
        (spells["txn_date"] >= expire)
        & (spells["txn_date"] <= lapse_confirm)
        & (spells["expire_date"] > expire)
    ).any()


def build_monthly_subscription_status(
    txn: pd.DataFrame, users_df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Return (monthly_status, user_panel_bounds).

    active: paid coverage overlaps the calendar month.
    paused: lapsed (no renewal within 30d) but resubscribes before OBS_END.
    churn: first lapse with no return; panel ends that month (absorbing).
    """
    txn = txn.merge(users_df[["msno", "join_date"]], on="msno")
    status_rows: list[dict] = []
    bounds_rows: list[dict] = []

    for msno, grp in txn.groupby("msno"):
        join_date = grp["join_date"].iloc[0]
        spells = grp[grp["is_cancel"] == 0].sort_values(["txn_date", "transaction_date"])
        intervals: list[tuple[pd.Timestamp, pd.Timestamp, str]] = []
        terminal_churn_tenure: int | None = None

        for row in spells.itertuples(index=False):
            intervals.append((row.txn_date, row.expire_date, "active"))

        for row in spells.itertuples(index=False):
            expire = row.expire_date
            if expire > OBS_END:
                continue
            lapse_confirm = min(expire + pd.Timedelta(days=RENEWAL_GRACE_DAYS), OBS_END)
            if _renewed_within_grace(spells, expire, lapse_confirm):
                continue

            later = spells[spells["txn_date"] > lapse_confirm]
            if len(later) and later.iloc[0]["txn_date"] <= OBS_END:
                pause_start = lapse_confirm + pd.Timedelta(days=1)
                pause_end = later.iloc[0]["txn_date"] - pd.Timedelta(days=1)
                if pause_start <= pause_end:
                    intervals.append((pause_start, pause_end, "paused"))
            else:
                churn_start = lapse_confirm + pd.Timedelta(days=1)
                if churn_start <= OBS_END:
                    intervals.append((churn_start, OBS_END, "churn"))
                terminal_churn_tenure = int(months_since_join(join_date, lapse_confirm).iloc[0])
                break

        max_tenure = int(months_since_join(join_date, OBS_END).iloc[0])
        panel_end = terminal_churn_tenure if terminal_churn_tenure is not None else max_tenure
        bounds_rows.append(
            {
                "msno": msno,
                "panel_end": panel_end,
                "first_churn_month": terminal_churn_tenure,
            }
        )

        for tenure in range(panel_end + 1):
            month_start, month_end = calendar_month_bounds(join_date, tenure)
            status: str | None = None
            for start, end, label in intervals:
                if not _month_overlaps(month_start, month_end, start, end):
                    continue
                if label == "active":
                    status = "active"
                elif label == "churn":
                    status = "churn"
                elif status != "churn":
                    status = "paused"
            status_rows.append(
                {"msno": msno, "user_tenure": tenure, "subscription_status": status or "active"}
            )

    return pd.DataFrame(status_rows), pd.DataFrame(bounds_rows)


monthly_status, user_panel_bounds = build_monthly_subscription_status(transactions, users)
users = users.merge(user_panel_bounds, on="msno", how="left")

print(monthly_status["subscription_status"].value_counts())
print(f"Users with terminal churn: {users['first_churn_month'].notna().sum():,}")
print(f"Users with at least one paused month: {monthly_status.loc[monthly_status.subscription_status == 'paused', 'msno'].nunique():,}")
monthly_status.head(10)

subscription_status
active    7089
paused     661
churn      454
Name: count, dtype: int64
Users with terminal churn: 533
Users with at least one paused month: 133


,msno,user_tenure,subscription_status
0,+7RV7kPwGK6I7pULHb+TkNV9hyVa8I8HzWhOa6llYv0=,0,active
1,+7RV7kPwGK6I7pULHb+TkNV9hyVa8I8HzWhOa6llYv0=,1,active
2,+7RV7kPwGK6I7pULHb+TkNV9hyVa8I8HzWhOa6llYv0=,2,churn
3,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,0,active
4,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,1,active
5,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,2,active
6,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,3,active
7,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,4,active
8,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,5,active
9,+AJ8SqViuJU6REYwcfcT/F3rHCbmZe8XKmI6CkKGhKk=,0,active


In [13]:
# --- monthly listening hours and cumulative hours ---

logs = logs.merge(users[["msno", "join_date"]], on="msno")
logs["user_tenure"] = months_since_join(logs["join_date"], logs["log_date"])
logs = logs[(logs["user_tenure"] >= 0) & (logs["log_date"] <= OBS_END)]

monthly_hours = (
    logs.groupby(["msno", "user_tenure"], as_index=False)["hours"].sum()
    .rename(columns={"hours": "monthly_hours"})
)
monthly_hours = monthly_hours.sort_values(["msno", "user_tenure"])
monthly_hours["cum_hours"] = monthly_hours.groupby("msno")["monthly_hours"].cumsum()

monthly_hours.head(10)

,msno,user_tenure,monthly_hours,cum_hours
0,+7RV7kPwGK6I7pULHb+TkNV9hyVa8I8HzWhOa6llYv0=,0,0.830490,0.830490
1,+7RV7kPwGK6I7pULHb+TkNV9hyVa8I8HzWhOa6llYv0=,1,0.342566,1.173057
2,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,0,9.933820,9.933820
3,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,1,32.097801,42.031621
4,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,2,14.021418,56.053039
5,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,3,31.607408,87.660447
6,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,4,11.151541,98.811988
7,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,5,13.285773,112.097761
8,+AJ8SqViuJU6REYwcfcT/F3rHCbmZe8XKmI6CkKGhKk=,0,7.473251,7.473251
9,+AJ8SqViuJU6REYwcfcT/F3rHCbmZe8XKmI6CkKGhKk=,1,43.004572,50.477823


In [14]:
# --- monthly subscription price and auto-renew (from renewals; forward-filled) ---

renewals = transactions[transactions["is_cancel"] == 0].copy()
renewals = renewals.merge(users[["msno", "join_date"]], on="msno")
renewals["user_tenure"] = months_since_join(renewals["join_date"], renewals["txn_date"])
renewals = renewals[renewals["user_tenure"] >= 0]

# If multiple renewals in a month, keep the last by transaction date
monthly_sub = (
    renewals.sort_values(["msno", "user_tenure", "transaction_date"])
    .groupby(["msno", "user_tenure"], as_index=False)
    .tail(1)[
        [
            "msno",
            "user_tenure",
            "plan_list_price",
            "is_auto_renew",
            "transaction_date",
            "membership_expire_date",
        ]
    ]
    .rename(
        columns={
            "plan_list_price": "price",
            "is_auto_renew": "auto_renewal",
        }
    )
)

monthly_sub.head(10)

,msno,user_tenure,price,auto_renewal,transaction_date,membership_expire_date
0,+7RV7kPwGK6I7pULHb+TkNV9hyVa8I8HzWhOa6llYv0=,0,149,1,20151222,20160122
1,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,0,99,1,20160914,20161014
2,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,1,99,1,20161014,20161114
3,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,2,99,1,20161114,20161214
4,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,3,99,1,20161214,20170114
5,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,4,99,1,20170114,20170214
6,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,5,99,1,20170214,20170314
7,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,6,99,1,20170314,20170414
8,+AJ8SqViuJU6REYwcfcT/F3rHCbmZe8XKmI6CkKGhKk=,0,149,0,20161229,20170131
9,+AJ8SqViuJU6REYwcfcT/F3rHCbmZe8XKmI6CkKGhKk=,2,149,0,20170204,20170306


In [15]:
# --- build user-month panel (truncate at terminal churn) ---

panel = monthly_status.merge(monthly_hours, on=["msno", "user_tenure"], how="left")
panel = panel.merge(monthly_sub, on=["msno", "user_tenure"], how="left")

panel = panel.sort_values(["msno", "user_tenure"])
for col in ("price", "auto_renewal"):
    panel[col] = panel.groupby("msno")[col].ffill()

panel["monthly_hours"] = panel["monthly_hours"].fillna(0.0)
panel["cum_hours"] = panel.groupby("msno")["monthly_hours"].cumsum()

panel.head(12)

,msno,user_tenure,subscription_status,monthly_hours,cum_hours,price,auto_renewal,transaction_date,membership_expire_date
0,+7RV7kPwGK6I7pULHb+TkNV9hyVa8I8HzWhOa6llYv0=,0,active,0.830490,0.830490,149.0,1.0,20151222.0,20160122.0
1,+7RV7kPwGK6I7pULHb+TkNV9hyVa8I8HzWhOa6llYv0=,1,active,0.342566,1.173057,149.0,1.0,NaN,NaN
2,+7RV7kPwGK6I7pULHb+TkNV9hyVa8I8HzWhOa6llYv0=,2,churn,0.000000,1.173057,149.0,1.0,NaN,NaN
3,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,0,active,9.933820,9.933820,99.0,1.0,20160914.0,20161014.0
4,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,1,active,32.097801,42.031621,99.0,1.0,20161014.0,20161114.0
5,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,2,active,14.021418,56.053039,99.0,1.0,20161114.0,20161214.0
6,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,3,active,31.607408,87.660447,99.0,1.0,20161214.0,20170114.0
7,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,4,active,11.151541,98.811988,99.0,1.0,20170114.0,20170214.0
8,+7mFvzKUae2FN6JHPLw+Bp5txSUwkFsr9FQM3epaihw=,5,active,13.285773,112.097761,99.0,1.0,20170214.0,20170314.0
9,+AJ8SqViuJU6REYwcfcT/F3rHCbmZe8XKmI6CkKGhKk=,0,active,7.473251,7.473251,149.0,0.0,20161229.0,20170131.0


In [17]:
# --- summary and save ---

out_path = SUBSAMPLE_DIR / "panel_user_month.csv"
panel.to_csv(out_path, index=False)

print(f"Saved panel -> {out_path}")
print(f"Rows: {len(panel):,}  |  Users: {panel.msno.nunique():,} (of {len(users):,} sampled)")
print(f"Mean months per user: {len(panel) / panel.msno.nunique():.1f}")
print("\nSubscription status:")
print(panel["subscription_status"].value_counts())

def active_after_churn(g: pd.DataFrame) -> bool:
    g = g.sort_values("user_tenure")
    churn_months = g.loc[g["subscription_status"] == "churn", "user_tenure"]
    if churn_months.empty:
        return False
    return ((g["subscription_status"] == "active") & (g["user_tenure"] > churn_months.min())).any()

print("\nMissing price after forward-fill:", panel["price"].isna().sum())

Saved panel -> kkbox-subsample/panel_user_month.csv
Rows: 8,204  |  Users: 998 (of 1,000 sampled)
Mean months per user: 8.2

Subscription status:
subscription_status
active    7089
paused     661
churn      454
Name: count, dtype: int64

Missing price after forward-fill: 9
